In [1]:
import sys

# Ultralytics provides the unified YOLO API (v8, v9, v10, v11)
!pip install ultralytics --upgrade

# Additional utilities
!pip install pandas matplotlib seaborn PyYAML

# Verify GPU availability
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ----------------- ---------------------- 0.5/1.2 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 3.7 MB/s  0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.19
    Uninstalling ultralytics-8.4.19:
      Successfully uninstalled ultralytics-8.4.19
PyTorch : 2.5.1+cu121
CUDA    : True
GPU     : NVIDIA GeForce RTX 4050 Laptop GPU
VRAM    : 6.4 GB


In [2]:
from ultralytics import YOLO
import ultralytics

print(f"Ultralytics version: {ultralytics.__version__}")

# Auto-downloads yolo11s.pt (~18 MB) from Ultralytics servers
model = YOLO('yolo11s.pt')
print("\n✅ YOLOv11s loaded successfully!")
print(model.info())  # Shows architecture summary


Ultralytics version: 8.4.21

✅ YOLOv11s loaded successfully!
YOLO11s summary: 181 layers, 9,458,752 parameters, 0 gradients, 21.7 GFLOPs
(181, 9458752, 0, 21.718374400000002)


In [3]:
import os

# ── Paths (same as your YOLOv5 setup) ─────────────────────────────────────
DATASET_DIR = r"C:\Users\Christian\AvianTech\data\datasets"
YAML_PATH   = r"C:\Users\Christian\AvianTech\config\birds.yaml"

# Verify dataset integrity
for split in ["train", "val", "test"]:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")
    imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f"  {split:<6} → {imgs} images | {lbls} labels")

print(f"\n✅ YAML config : {YAML_PATH}")
print(f"✅ Dataset root: {DATASET_DIR}")

  train  → 1294 images | 1292 labels
  val    → 370 images | 370 labels
  test   → 185 images | 185 labels

✅ YAML config : C:\Users\Christian\AvianTech\config\birds.yaml
✅ Dataset root: C:\Users\Christian\AvianTech\data\datasets


In [4]:
import yaml

yaml_content = {
    "path": r"C:\Users\Christian\AvianTech\data\datasets",
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc": 37,
    "names": [
        "Amethyst brown dove",
        "Asian Glossy Starling",
        "Blue-capped Kingfisher",
        "Blue-crowned Racquet-tail",
        "Blue-naped Parrot",
        "Brown Shrike",
        "Celestial Monarch",
        "Chestnut Munia",
        "Coleto",
        "Eurasian Tree Sparrow",
        "Garden Sunbird",
        "Golden-bellied Gerygone",
        "Green imperial pigeon",
        "Guaiabero",
        "Large-billed crow",
        "Mindanao Boobook",
        "Mindanao Heleia",
        "Mindanao Scops-Owl",
        "Philippine Bulbul",
        "Philippine Coucal",
        "Philippine Eagle",
        "Philippine Hanging-Parrot",
        "Philippine Pied-Fantail",
        "Philippine Serpent Eagle",
        "Philippine Trogon",
        "Philippine cuckoo-dove",
        "Ridgetop Swiflet",
        "Rock Pigeon",
        "Rufous Hornbill",
        "Striated Grass Warbler",
        "Striated Wren",
        "White-eared Brown Dove",
        "White-throated Kingfisher",
        "Yellow-vented Bulbul",
        "Yellow-wattled Bulbul",
        "Zebra Dove",
        "red-keeled flower pecker"
    ]
}

with open(YAML_PATH, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)

print("✅ birds.yaml verified and ready!")
print(f"🐦 Total species: {yaml_content['nc']}")

✅ birds.yaml verified and ready!
🐦 Total species: 37


In [5]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')  # Load pretrained backbone

results = model.train(
    data    = r"C:\Users\Christian\AvianTech\config\birds.yaml",
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,           # Reduce to 8 if you get OOM errors
    device  = 0,            # GPU 0
    project = r"C:\Users\Christian\AvianTech\models\yolov11",
    name    = "aviantech_v11_phase1",
    pretrained = True,
    optimizer  = "AdamW",   # Better convergence than SGD for fine-grained classes
    lr0        = 0.001,
    lrf        = 0.01,
    momentum   = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    cos_lr     = True,      # Cosine LR schedule
    augment    = True,
    hsv_h      = 0.015,     # Hue augmentation (good for plumage color variation)
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    flipud     = 0.1,
    fliplr     = 0.5,
    mosaic     = 1.0,
    mixup      = 0.1,
    copy_paste = 0.1,
    patience   = 15,        # Early stopping
    save       = True,
    exist_ok   = True,
    verbose    = True,
)

print("\n✅ Phase 1 training complete!")
print(f"Best mAP@0.5 : {results.results_dict['metrics/mAP50(B)']:.4f}")

Ultralytics 8.4.21  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Christian\AvianTech\config\birds.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aviantech_v11_phase1, nbs=64, nms=False, opset=None, optimize=False, optimizer=

In [6]:
from ultralytics import YOLO

# Load best weights from Phase 1
best_weights = r"C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase1\weights\best.pt"
model = YOLO(best_weights)

results = model.train(
    data    = r"C:\Users\Christian\AvianTech\config\birds.yaml",
    epochs  = 100,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    project = r"C:\Users\Christian\AvianTech\models\yolov11",
    name    = "aviantech_v11_phase2",
    pretrained = True,
    optimizer  = "AdamW",
    lr0        = 0.0005,    # Lower LR for fine-tuning
    lrf        = 0.001,
    cos_lr     = True,
    patience   = 20,
    save       = True,
    exist_ok   = True,
    verbose    = True,
)

print("\n✅ Phase 2 fine-tuning complete!")
print(f"Best mAP@0.5      : {results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"Best mAP@0.5:0.95 : {results.results_dict['metrics/mAP50-95(B)']:.4f}")

Ultralytics 8.4.21  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Christian\AvianTech\config\birds.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase1\weights\best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=avian

In [7]:
from ultralytics import YOLO

best_weights = r"C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase2\weights\best.pt"
model = YOLO(best_weights)

metrics = model.val(
    data   = r"C:\Users\Christian\AvianTech\config\birds.yaml",
    split  = "test",
    device = 0,
    verbose = True,
    plots  = True,          # Saves confusion matrix, PR curve, etc.
)

print("\n📊 Test Set Results:")
print(f"  mAP@0.5       : {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95  : {metrics.box.map:.4f}")
print(f"  Precision     : {metrics.box.mp:.4f}")
print(f"  Recall        : {metrics.box.mr:.4f}")

Ultralytics 8.4.21  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11s summary (fused): 101 layers, 9,427,119 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.50.4 ms, read: 31.79.8 MB/s, size: 39.1 KB)
val: Scanning C:\Users\Christian\AvianTech\data\datasets\test\labels... 185 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 185/185 1.3Kit/s 0.1s<0.2s
val: New cache created: C:\Users\Christian\AvianTech\data\datasets\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 4.9it/s 2.4s0.2s
                   all        185        290      0.906      0.775      0.855      0.746
   Amethyst brown dove          5          8      0.842      0.669      0.752      0.607
 Asian Glossy Starling          5          8      0.731      0.875        0.8       0.78
Blue-capped Kingfisher          9         13      0.964      0.692      0.817      0.791
Blue-c

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import os

results_csv = r"C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase2\results.csv"
save_dir    = r"C:\Users\Christian\AvianTech\results_charts_v11"
os.makedirs(save_dir, exist_ok=True)

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
epochs = df["epoch"]

# ── Style ──────────────────────────────────────────────────────────────────
BG, CARD = "#0f1117", "#1a1d27"
GRN, ACC, YLW, PRP, RED = "#00ff88", "#00d4ff", "#ffa502", "#a29bfe", "#ff4757"
WHT, GRY = "#ffffff", "#8892a4"

best_idx = df["metrics/mAP50(B)"].idxmax()
best     = df.loc[best_idx]

def base_fig(title):
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
    for sp in ax.spines.values(): sp.set_edgecolor("#2a2d3a")
    ax.tick_params(colors=GRY, labelsize=9)
    ax.set_title(title, color=WHT, fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Epoch", color=GRY, fontsize=10)
    ax.grid(color="#2a2d3a", linewidth=0.5)
    ax.axvline(x=best["epoch"], color=YLW, linestyle="--",
               linewidth=1.2, alpha=0.7, label=f"Best epoch ({int(best['epoch'])})")
    return fig, ax

def save(fig, name):
    path = os.path.join(save_dir, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show(); plt.close(fig)
    print(f"✅ Saved → {path}")

# 1. mAP Curves
fig, ax = base_fig("mAP Curves — YOLOv11s")
ax.plot(epochs, df["metrics/mAP50(B)"],       color=GRN, linewidth=2.5, label=f"mAP@0.5  (best={best['metrics/mAP50(B)']:.4f})")
ax.plot(epochs, df["metrics/mAP50-95(B)"],    color=ACC, linewidth=2.5, label=f"mAP@0.5:0.95  (best={best['metrics/mAP50-95(B)']:.4f})")
ax.fill_between(epochs, df["metrics/mAP50(B)"],    alpha=0.12, color=GRN)
ax.fill_between(epochs, df["metrics/mAP50-95(B)"], alpha=0.12, color=ACC)
ax.set_ylabel("mAP", color=GRY, fontsize=10)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=9)
save(fig, "1_mAP_curves.png")

# 2. Precision & Recall
fig, ax = base_fig("Precision & Recall — YOLOv11s")
ax.plot(epochs, df["metrics/precision(B)"], color=YLW, linewidth=2.5, label=f"Precision  (best={best['metrics/precision(B)']:.4f})")
ax.plot(epochs, df["metrics/recall(B)"],    color=PRP, linewidth=2.5, label=f"Recall  (best={best['metrics/recall(B)']:.4f})")
ax.fill_between(epochs, df["metrics/precision(B)"], alpha=0.12, color=YLW)
ax.fill_between(epochs, df["metrics/recall(B)"],    alpha=0.12, color=PRP)
ax.set_ylabel("Score", color=GRY, fontsize=10)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=9)
save(fig, "2_precision_recall.png")

# 3. Training Losses
fig, ax = base_fig("Training Losses — YOLOv11s")
ax.plot(epochs, df["train/box_loss"], color=RED, linewidth=2.5, label="Box Loss")
ax.plot(epochs, df["train/cls_loss"], color=PRP, linewidth=2.5, label="Class Loss")
ax.plot(epochs, df["train/dfl_loss"], color=YLW, linewidth=2.5, label="DFL Loss")  # YOLOv11 uses DFL
ax.fill_between(epochs, df["train/box_loss"], alpha=0.10, color=RED)
ax.fill_between(epochs, df["train/cls_loss"], alpha=0.10, color=PRP)
ax.fill_between(epochs, df["train/dfl_loss"], alpha=0.10, color=YLW)
ax.set_ylabel("Loss", color=GRY, fontsize=10)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=9)
save(fig, "3_train_losses.png")

# 4. Validation Losses
fig, ax = base_fig("Validation Losses — YOLOv11s")
ax.plot(epochs, df["val/box_loss"], color=RED, linewidth=2.5, label="Val Box Loss")
ax.plot(epochs, df["val/cls_loss"], color=PRP, linewidth=2.5, label="Val Class Loss")
ax.plot(epochs, df["val/dfl_loss"], color=YLW, linewidth=2.5, label="Val DFL Loss")
ax.fill_between(epochs, df["val/box_loss"], alpha=0.10, color=RED)
ax.fill_between(epochs, df["val/cls_loss"], alpha=0.10, color=PRP)
ax.fill_between(epochs, df["val/dfl_loss"], alpha=0.10, color=YLW)
ax.set_ylabel("Loss", color=GRY, fontsize=10)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=9)
save(fig, "4_val_losses.png")

# 5. F1 Score
p  = df["metrics/precision(B)"]
r  = df["metrics/recall(B)"]
f1 = 2 * p * r / (p + r + 1e-8)
fig, ax = base_fig("F1 Score — YOLOv11s")
ax.plot(epochs, f1, color=RED, linewidth=2.5, label=f"F1  (best={f1.max():.4f})")
ax.fill_between(epochs, f1, alpha=0.12, color=RED)
ax.set_ylabel("F1 Score", color=GRY, fontsize=10)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=9)
save(fig, "5_f1_score.png")

print(f"\n🎉 All charts saved to:\n   {save_dir}")

<Figure size 1000x500 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v11\1_mAP_curves.png


<Figure size 1000x500 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v11\2_precision_recall.png


<Figure size 1000x500 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v11\3_train_losses.png


<Figure size 1000x500 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v11\4_val_losses.png


<Figure size 1000x500 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v11\5_f1_score.png

🎉 All charts saved to:
   C:\Users\Christian\AvianTech\results_charts_v11


In [9]:
import pandas as pd

# ── Load both results CSVs ──────────────────────────────────────────────────
v5_csv  = r"C:\Users\Christian\AvianTech\models\yolov5\aviantech_birds_v2\results.csv"
v11_csv = r"C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase2\results.csv"

df5  = pd.read_csv(v5_csv);  df5.columns  = df5.columns.str.strip()
df11 = pd.read_csv(v11_csv); df11.columns = df11.columns.str.strip()

# Best epoch from each
best5  = df5.loc[df5["metrics/mAP_0.5"].idxmax()]
best11 = df11.loc[df11["metrics/mAP50(B)"].idxmax()]

# ── Summary Table ───────────────────────────────────────────────────────────
print("=" * 60)
print(f"{'Metric':<22} {'YOLOv5s':>15} {'YOLOv11s':>15}")
print("=" * 60)
print(f"{'mAP@0.5':<22} {best5['metrics/mAP_0.5']:>15.4f} {best11['metrics/mAP50(B)']:>15.4f}")
print(f"{'mAP@0.5:0.95':<22} {best5['metrics/mAP_0.5:0.95']:>15.4f} {best11['metrics/mAP50-95(B)']:>15.4f}")
print(f"{'Precision':<22} {best5['metrics/precision']:>15.4f} {best11['metrics/precision(B)']:>15.4f}")
print(f"{'Recall':<22} {best5['metrics/recall']:>15.4f} {best11['metrics/recall(B)']:>15.4f}")
print(f"{'Best Epoch':<22} {int(best5['epoch']):>15} {int(best11['epoch']):>15}")
print("=" * 60)

# Improvement
map50_diff = best11['metrics/mAP50(B)'] - best5['metrics/mAP_0.5']
print(f"\n📈 mAP@0.5 improvement: {map50_diff:+.4f} ({map50_diff*100:+.2f}%)")

Metric                         YOLOv5s        YOLOv11s
mAP@0.5                         0.7572          0.8039
mAP@0.5:0.95                    0.6056          0.6893
Precision                       0.8155          0.8236
Recall                          0.6740          0.7120
Best Epoch                          90              88

📈 mAP@0.5 improvement: +0.0466 (+4.66%)


In [10]:
from ultralytics import YOLO

best_weights = r"C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase2\weights\best.pt"
model = YOLO(best_weights)

# ── Option A: NCNN (recommended for Raspberry Pi 5) ────────────────────────
model.export(
    format  = "ncnn",
    imgsz   = 640,
    half    = False,   # Pi 5 doesn't support FP16 natively
    dynamic = False,
)
print("✅ NCNN export complete — deploy to Raspberry Pi 5")

# ── Option B: ONNX (cross-platform fallback) ───────────────────────────────
model.export(
    format  = "onnx",
    imgsz   = 640,
    half    = False,
    dynamic = False,
    simplify = True,
)
print("✅ ONNX export complete")

# ── Option C: TFLite (for TensorFlow Lite on Pi) ───────────────────────────
model.export(
    format = "tflite",
    imgsz  = 640,
    half   = False,
    int8   = False,   # Set True for INT8 quantization (faster, slightly less accurate)
)
print("✅ TFLite export complete")

Ultralytics 8.4.21  Python-3.12.10 torch-2.5.1+cu121 CPU (13th Gen Intel Core i5-13500HX)
WARNING NCNN export does not support end2end models, disabling end2end branch.
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s summary (fused): 101 layers, 9,427,119 parameters, 0 gradients, 21.4 GFLOPs

PyTorch: starting from 'C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase2\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 41, 8400) (18.3 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.12.10 environment at: C:\Users\Christian\cv_env
Resolved 1 package in 343ms
Prepared 1 package in 1.36s
Installed 1 package in 8ms
 + ncnn==1.0.20260114

requirements: AutoUpdate success  6.8s
WARNING requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirement ['pn

ModuleNotFoundError: No module named 'onnx2tf'

In [11]:
from ultralytics import YOLO
import cv2

best_weights = r"C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase2\weights\best.pt"
model = YOLO(best_weights)

# ── On a single image ──────────────────────────────────────────────────────
results = model.predict(
    source     = r"C:\Users\Christian\AvianTech\data\datasets\test\images",
    conf       = 0.25,      # Confidence threshold
    iou        = 0.45,      # NMS IoU threshold
    imgsz      = 640,
    device     = 0,
    save       = True,
    save_txt   = True,      # Save labels
    save_conf  = True,
    project    = r"C:\Users\Christian\AvianTech\inference",
    name       = "v11_test_run",
    show_labels = True,
    show_conf   = True,
    line_width  = 2,
)

# Print detections for first image
for r in results[:3]:
    print(f"\n📸 Image: {r.path}")
    for box in r.boxes:
        cls_id = int(box.cls)
        conf   = float(box.conf)
        name   = model.names[cls_id]
        print(f"   🐦 {name:<35} conf={conf:.2%}")


image 1/185 C:\Users\Christian\AvianTech\data\datasets\test\images\480 (2)_jpg.rf.F9eY16cxrncT8FUGAnIf.jpg: 512x640 2 Celestial Monarchs, 64.2ms
image 2/185 C:\Users\Christian\AvianTech\data\datasets\test\images\480 (28)_jpg.rf.OO5zc8zRlgleXd1OTxcO.jpg: 512x640 2 Celestial Monarchs, 22.4ms
image 3/185 C:\Users\Christian\AvianTech\data\datasets\test\images\480 (6)_jpg.rf.A0aZKslkOLMj1Jl5zLsA.jpg: 448x640 1 White-throated Kingfisher, 72.1ms
image 4/185 C:\Users\Christian\AvianTech\data\datasets\test\images\480 (8)_jpg.rf.fI2GhIUYMlpexo1NYo5m.jpg: 640x512 1 White-throated Kingfisher, 61.9ms
image 5/185 C:\Users\Christian\AvianTech\data\datasets\test\images\Asian Glossy Starling (12)_jpg.rf.aVBX6kyaFee1iyd7SKSn.jpg: 448x640 1 Asian Glossy Starling, 24.8ms
image 6/185 C:\Users\Christian\AvianTech\data\datasets\test\images\Asian Glossy Starling (27)_jpg.rf.XEFLPmwlNn7EkNljmvrY.jpg: 480x640 2 Asian Glossy Starlings, 52.5ms
image 7/185 C:\Users\Christian\AvianTech\data\datasets\test\images\As

In [12]:
from ultralytics import YOLO
import time, torch

best_weights = r"C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase2\weights\best.pt"
model = YOLO(best_weights)

# Benchmark inference speed
results = model.benchmark(
    data   = r"C:\Users\Christian\AvianTech\config\birds.yaml",
    imgsz  = 640,
    half   = False,
    device = 0,
)
print(results)

Setup complete  (20 CPUs, 15.7 GB RAM, 306.9/475.7 GB disk)

Benchmarks complete for C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase2\weights\best.pt on C:\Users\Christian\AvianTech\config\birds.yaml at imgsz=640 (634.96s)
Benchmarks legend:  -  Success  -  Export passed but validation failed  -  Export failed
+-----------------------------------------------------------------------------------------------------------+
|      Format                  Status   Size (MB)   metrics/mAP50-95(B)   Inference time (ms/im)   FPS    |
+===========================================================================================================+
| 1    PyTorch                          18.3        0.6888                12.55                    79.69  |
| 2    TorchScript                      36.5        0.6852                12.45                    80.35  |
| 3    ONNX                             36.2        0.6856                8.78                     113.94 |
| 4    OpenVINO     

In [15]:
# Fixed version check
try:
    from birdnetlib import Recording
    from birdnetlib.analyzer import Analyzer
    import librosa
    print(f"✅ birdnetlib  : installed and working")
    print(f"✅ librosa     : {librosa.__version__}")
except ImportError as e:
    print(f"❌ Missing: {e}")

✅ birdnetlib  : installed and working
✅ librosa     : 0.11.0


In [16]:
import os

AUDIO_BASE = r"C:\Users\Christian\AvianTech\Birds Audio"

print("🎵 Audio dataset structure:\n")
total_files  = 0
species_list = []

for folder in sorted(os.listdir(AUDIO_BASE)):
    folder_path = os.path.join(AUDIO_BASE, folder)
    if not os.path.isdir(folder_path):
        continue
    files = [f for f in os.listdir(folder_path)
             if f.lower().endswith(('.wav', '.mp3', '.ogg', '.flac', '.m4a'))]
    total_files += len(files)
    species_list.append((folder, len(files)))
    print(f"  📁 {folder:<40} {len(files):>3} file(s)")

print(f"\n{'─'*55}")
print(f"  Total species  : {len(species_list)}")
print(f"  Total files    : {total_files}")
print(f"  Audio base dir : {AUDIO_BASE}")

🎵 Audio dataset structure:

  📁 Amethyst Brown Dove (Phapitreron amethystinus)   1 file(s)
  📁 Asian Glossy Starling (Aplonis panayensis)   1 file(s)
  📁 Blue-capped Kingfisher (Actenoides hombroni)   1 file(s)
  📁 Blue-crowned Racquet-tail (Prioniturus discurus)   1 file(s)
  📁 Blue-naped Parrot (Tanygnathus lucionensis)   1 file(s)
  📁 Brown Shrike (Lanius cristatus)            1 file(s)
  📁 Celestial Monarch (Hypothymis coelestis)   1 file(s)
  📁 Chestnut Munia (Lonchura atricapilla)      1 file(s)
  📁 Coleto (Sarcops calvus)                    1 file(s)
  📁 Eurasian Tree Sparrow (Passer montanus)    1 file(s)
  📁 Garden Sunbird (Cinnyris jugularis)        1 file(s)
  📁 Golden-bellied Gerygone (Gerygone sulphurea)   1 file(s)
  📁 Green Imperial Pigeon (Ducula aenea)       1 file(s)
  📁 Guaiabero (Bolbopsittacus lunulatus)       1 file(s)
  📁 Large-billed Crow (Corvus macrorhynchos)   1 file(s)
  📁 Mindanao Boobook (Ninox Spilocephala)      1 file(s)
  📁 Mindanao Heleia (Heleia goodf

In [17]:
import os
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer

AUDIO_BASE = r"C:\Users\Christian\AvianTech\Birds Audio"

# Initialize BirdNET — uses Iligan City coordinates to bias toward local species
analyzer = Analyzer()

results_summary = []
detected_count  = 0

print("🎤 Running BirdNET on all 37 species...\n")
print(f"{'Species':<42} {'Top Detection':<35} {'Conf':>6}  {'Match'}")
print("─" * 95)

for species_folder in sorted(os.listdir(AUDIO_BASE)):
    folder_path = os.path.join(AUDIO_BASE, species_folder)
    if not os.path.isdir(folder_path):
        continue

    audio_files = [f for f in os.listdir(folder_path)
                   if f.lower().endswith(('.wav', '.mp3', '.ogg', '.flac', '.m4a'))]
    if not audio_files:
        continue

    # Run on first available audio file per species
    audio_file = os.path.join(folder_path, audio_files[0])

    try:
        recording = Recording(
            analyzer,
            audio_file,
            lat     = 8.2280,    # Iligan City latitude
            lon     = 124.2452,  # Iligan City longitude
            min_conf = 0.1,
        )
        recording.analyze()

        if recording.detections:
            top      = recording.detections[0]
            detected = top['common_name']
            conf     = top['confidence']
            # Check if BirdNET's top detection matches the folder species name
            match    = "✅" if species_folder.lower() in detected.lower() or \
                               detected.lower() in species_folder.lower() else "⚠️"
            if match == "✅":
                detected_count += 1
        else:
            detected = "No detection"
            conf     = 0.0
            match    = "❌"

        print(f"  {species_folder[:40]:<40}  {detected[:33]:<35} {conf:>5.1%}  {match}")
        results_summary.append({
            "species"       : species_folder,
            "detected"      : detected,
            "confidence"    : conf,
            "match"         : match,
            "audio_file"    : audio_files[0],
            "total_files"   : len(audio_files),
        })

    except Exception as e:
        print(f"  ⚠️  {species_folder[:40]:<40}  Error: {e}")

print(f"\n{'─'*95}")
print(f"  ✅ Matched  : {detected_count} / {len(results_summary)} species")
print(f"  📊 Done! Tested {len(results_summary)} / 37 species")

Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.
🎤 Running BirdNET on all 37 species...

Species                                    Top Detection                         Conf  Match
───────────────────────────────────────────────────────────────────────────────────────────────
read_audio_data


C:\Users\Christian\cv_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
C:\Users\Christian\cv_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


read_audio_data: complete, read  7 chunks.
analyze_recording XC963886 - Amethyst Brown Dove - Phapitreron amethystinus amethystinus.wav
recording has lon/lat
set_predicted_species_list_from_position
return_predicted_species_list
-1
277 species loaded.
  Amethyst Brown Dove (Phapitreron amethys  Amethyst Brown-Dove                 14.7%  ⚠️
read_audio_data
read_audio_data: complete, read  32 chunks.
analyze_recording Asian Glossy Starling - Aplonis panayensis.wav
recording has lon/lat
set_predicted_species_list_from_position
  Asian Glossy Starling (Aplonis panayensi  Warbling White-eye                  15.3%  ⚠️
read_audio_data
read_audio_data: complete, read  12 chunks.
analyze_recording Kingfisher - Actenoides hombroni.mp3
recording has lon/lat
set_predicted_species_list_from_position
  Blue-capped Kingfisher (Actenoides hombr  Philippine Oriole                   34.9%  ⚠️
read_audio_data
read_audio_data: complete, read  28 chunks.
analyze_recording XC965153 - Blue-crowned Racket-tai

In [18]:
import pandas as pd

df_audio = pd.DataFrame(results_summary)

# Sort by confidence descending
df_audio = df_audio.sort_values("confidence", ascending=False).reset_index(drop=True)

# Display full table
pd.set_option("display.max_rows",    50)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width",       120)

print("📋 BirdNET Classification Results — 37 Species\n")
print(df_audio[["species", "detected", "confidence", "match", "total_files"]].to_string(index=False))

print(f"\n{'─'*60}")
print(f"  Total tested  : {len(df_audio)}")
print(f"  Matched (✅)  : {(df_audio['match'] == '✅').sum()}")
print(f"  Partial  (⚠️) : {(df_audio['match'] == '⚠️').sum()}")
print(f"  No detect(❌) : {(df_audio['match'] == '❌').sum()}")
print(f"  Avg confidence: {df_audio['confidence'].mean():.2%}")

📋 BirdNET Classification Results — 37 Species

                                           species                  detected  confidence match  total_files
      Golden-bellied Gerygone (Gerygone sulphurea)   Golden-bellied Gerygone    0.998742     ✅            1
               Garden Sunbird (Cinnyris jugularis)      Olive-backed Sunbird    0.997519    ⚠️            1
  Philippine Pied-Fantail (Rhipidura nigritorquis)   Philippine Pied-Fantail    0.976833     ✅            1
              Philippine Trogon (Harpactes ardens)         Philippine Trogon    0.974687     ✅            1
              Philippine Coucal(Centropus viridis)         Philippine Coucal    0.974051     ✅            1
    Yellow-wattled Bulbul (Microtarsus urostictus)     Yellow-wattled Bulbul    0.936647     ✅            1
 Striated Wren (Babbler - Ptilocichla mindanensis)     Striated Wren-Babbler    0.902584    ⚠️            1
                       Rock Pigeon (Columba livia)               Rock Pigeon    0.899085 

In [19]:
import matplotlib.pyplot as plt
import numpy as np
import os

save_dir = r"C:\Users\Christian\AvianTech\results_charts_v11"
os.makedirs(save_dir, exist_ok=True)

# ── Style (matches existing notebook charts) ───────────────────────────────
BG, CARD = "#0f1117", "#1a1d27"
GRN, ACC, YLW, RED = "#00ff88", "#00d4ff", "#ffa502", "#ff4757"
WHT, GRY = "#ffffff", "#8892a4"

df_plot = df_audio.sort_values("confidence", ascending=True)
colors  = [GRN if m == "✅" else YLW if m == "⚠️" else RED
           for m in df_plot["match"]]

fig, ax = plt.subplots(figsize=(12, 14))
fig.patch.set_facecolor(BG)
ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2d3a")

bars = ax.barh(df_plot["species"], df_plot["confidence"],
               color=colors, edgecolor="#2a2d3a", height=0.7)

# Confidence value labels
for bar, conf in zip(bars, df_plot["confidence"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{conf:.1%}", va="center", ha="left", fontsize=8, color=WHT)

# Reference line at 0.5
ax.axvline(x=0.5, color=ACC, linestyle="--", linewidth=1.2, alpha=0.7,
           label="0.5 confidence threshold")

ax.set_xlabel("BirdNET Confidence Score", color=GRY, fontsize=11)
ax.set_title("BirdNET Audio Classification — 37 Philippine Bird Species",
             color=WHT, fontsize=14, fontweight="bold", pad=14)
ax.tick_params(colors=GRY, labelsize=9)
ax.set_xlim(0, 1.1)
ax.grid(axis="x", color="#2a2d3a", linewidth=0.5)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=GRN, label="✅ Species matched"),
    Patch(facecolor=YLW, label="⚠️  Partial match"),
    Patch(facecolor=RED, label="❌ No detection"),
]
ax.legend(handles=legend_elements, facecolor=CARD, edgecolor=GRY,
          labelcolor=WHT, fontsize=10, loc="lower right")

plt.tight_layout()
out_path = os.path.join(save_dir, "birdnet_confidence_chart.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"✅ Saved → {out_path}")

C:\Users\Christian\AppData\Local\Temp\ipykernel_12984\1909832477.py:52: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\Christian\AppData\Local\Temp\ipykernel_12984\1909832477.py:52: UserWarning: Glyph 10060 (\N{CROSS MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\Christian\AppData\Local\Temp\ipykernel_12984\1909832477.py:54: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=BG)
C:\Users\Christian\AppData\Local\Temp\ipykernel_12984\1909832477.py:54: UserWarning: Glyph 10060 (\N{CROSS MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=BG)


<Figure size 1200x1400 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v11\birdnet_confidence_chart.png


In [20]:
import pandas as pd
import numpy as np

# ── YOLOv11 per-class results (from val output) ────────────────────────────
# Load the per-class precision/recall from YOLOv11 validation
# Ultralytics saves this in the run directory as a verbose val output
# We reconstruct it from the YOLO metrics object stored earlier

from ultralytics import YOLO

best_weights = r"C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase2\weights\best.pt"
model        = YOLO(best_weights)

metrics = model.val(
    data    = r"C:\Users\Christian\AvianTech\config\birds.yaml",
    split   = "test",
    device  = 0,
    verbose = False,
)

# Per-class AP50 scores
class_names = list(model.names.values())   # 37 species names
ap50_scores = metrics.box.ap50             # numpy array, one per class

df_yolo = pd.DataFrame({
    "species"      : class_names,
    "yolo_ap50"    : ap50_scores,
})

# ── Merge with BirdNET results ─────────────────────────────────────────────
# Normalize species names for joining (lowercase, strip)
df_yolo["key"]  = df_yolo["species"].str.lower().str.strip()
df_audio["key"] = df_audio["species"].str.lower().str.strip()

df_fused = pd.merge(df_yolo, df_audio[["key", "confidence", "match"]],
                    on="key", how="left")
df_fused = df_fused.rename(columns={"confidence": "birdnet_conf"})
df_fused["birdnet_conf"] = df_fused["birdnet_conf"].fillna(0.0)

# ── Fusion score: weighted average (60% YOLO, 40% BirdNET) ────────────────
YOLO_WEIGHT    = 0.6
BIRDNET_WEIGHT = 0.4

df_fused["fusion_score"] = (
    df_fused["yolo_ap50"]    * YOLO_WEIGHT +
    df_fused["birdnet_conf"] * BIRDNET_WEIGHT
)

df_fused = df_fused.sort_values("fusion_score", ascending=False).reset_index(drop=True)
df_fused.index += 1   # rank from 1

# ── Print table ────────────────────────────────────────────────────────────
print("🔗 AvianTech — Fused Detection Results (YOLOv11 + BirdNET)\n")
print(f"{'#':<4} {'Species':<38} {'YOLO AP50':>10} {'BirdNET':>9} {'Fusion':>9}  {'Audio'}")
print("─" * 85)

for rank, row in df_fused.iterrows():
    print(f"  {rank:<3} {row['species']:<38} "
          f"{row['yolo_ap50']:>9.1%} "
          f"{row['birdnet_conf']:>8.1%} "
          f"{row['fusion_score']:>8.1%}  "
          f"{row['match']}")

print(f"\n{'─'*85}")
print(f"  Avg YOLO AP50     : {df_fused['yolo_ap50'].mean():.2%}")
print(f"  Avg BirdNET conf  : {df_fused['birdnet_conf'].mean():.2%}")
print(f"  Avg Fusion score  : {df_fused['fusion_score'].mean():.2%}")

Ultralytics 8.4.21  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11s summary (fused): 101 layers, 9,427,119 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 98.832.9 MB/s, size: 45.5 KB)
val: Scanning C:\Users\Christian\AvianTech\data\datasets\test\labels.cache... 185 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 185/185  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 4.3it/s 2.8s0.2s
                   all        185        290      0.906      0.775      0.855      0.746
Speed: 1.4ms preprocess, 9.0ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to C:\Users\Christian\AvianTech\runs\detect\val6
🔗 AvianTech — Fused Detection Results (YOLOv11 + BirdNET)

#    Species                                 YOLO AP50   BirdNET    Fusion  Audio
──────────────────────────────────────────────────────────────────────────────

In [21]:
import matplotlib.pyplot as plt
import os

save_dir = r"C:\Users\Christian\AvianTech\results_charts_v11"

BG, CARD = "#0f1117", "#1a1d27"
GRN, ACC, YLW, PRP = "#00ff88", "#00d4ff", "#ffa502", "#a29bfe"
WHT, GRY = "#ffffff", "#8892a4"

df_plot = df_fused.sort_values("fusion_score", ascending=True)
y       = np.arange(len(df_plot))
h       = 0.28

fig, ax = plt.subplots(figsize=(13, 15))
fig.patch.set_facecolor(BG)
ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2d3a")

# Three bars per species: YOLO, BirdNET, Fusion
ax.barh(y + h,  df_plot["yolo_ap50"],    height=h, color=GRN, label="YOLO AP50",    alpha=0.9)
ax.barh(y,      df_plot["birdnet_conf"], height=h, color=ACC, label="BirdNET Conf", alpha=0.9)
ax.barh(y - h,  df_plot["fusion_score"], height=h, color=YLW, label="Fusion Score", alpha=0.9)

ax.set_yticks(y)
ax.set_yticklabels(df_plot["species"], fontsize=8.5, color=WHT)
ax.set_xlabel("Score", color=GRY, fontsize=11)
ax.set_title("AvianTech — YOLOv11 + BirdNET Fusion Score per Species",
             color=WHT, fontsize=14, fontweight="bold", pad=14)
ax.tick_params(colors=GRY, labelsize=9)
ax.set_xlim(0, 1.15)
ax.axvline(x=0.5, color="#2a2d3a", linestyle="--", linewidth=1, alpha=0.6)
ax.grid(axis="x", color="#2a2d3a", linewidth=0.5)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=10)

plt.tight_layout()
out_path = os.path.join(save_dir, "fusion_score_chart.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"✅ Saved → {out_path}")

<Figure size 1300x1500 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v11\fusion_score_chart.png


In [22]:
import pandas as pd
import os

save_dir  = r"C:\Users\Christian\AvianTech\results_charts_v11"
xlsx_path = os.path.join(save_dir, "aviantech_full_results.xlsx")

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:

    # Sheet 1: Fusion results
    df_fused.drop(columns=["key"]).to_excel(
        writer, sheet_name="Fusion Results", index=True)

    # Sheet 2: BirdNET only
    df_audio[["species", "detected", "confidence", "match",
              "audio_file", "total_files"]].to_excel(
        writer, sheet_name="BirdNET Results", index=False)

    # Sheet 3: YOLO only
    df_yolo[["species", "yolo_ap50"]].to_excel(
        writer, sheet_name="YOLOv11 Per-Class AP50", index=False)

print(f"✅ Results saved to Excel → {xlsx_path}")
print("\n  Sheets:")
print("    📄 Fusion Results         — combined YOLO + BirdNET scores")
print("    📄 BirdNET Results        — raw audio classification output")
print("    📄 YOLOv11 Per-Class AP50 — per-species visual detection scores")

✅ Results saved to Excel → C:\Users\Christian\AvianTech\results_charts_v11\aviantech_full_results.xlsx

  Sheets:
    📄 Fusion Results         — combined YOLO + BirdNET scores
    📄 BirdNET Results        — raw audio classification output
    📄 YOLOv11 Per-Class AP50 — per-species visual detection scores


In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

save_dir = r"C:\Users\Christian\AvianTech\results_charts_v11"
os.makedirs(save_dir, exist_ok=True)

# ── Style ──────────────────────────────────────────────────────────────────
BG, CARD = "#0f1117", "#1a1d27"
GRN, ACC, YLW, PRP, RED = "#00ff88", "#00d4ff", "#ffa502", "#a29bfe", "#ff4757"
WHT, GRY, BLU = "#ffffff", "#8892a4", "#1e90ff"

# ══════════════════════════════════════════════════════════════════════════
# STEP 21 — Side-by-Side Model Metrics: Without Audio vs With Audio
# ══════════════════════════════════════════════════════════════════════════

# ── Metrics WITHOUT audio (YOLOv11 only) ──────────────────────────────────
from ultralytics import YOLO

best_weights = r"C:\Users\Christian\AvianTech\models\yolov11\aviantech_v11_phase2\weights\best.pt"
model   = YOLO(best_weights)
metrics = model.val(
    data    = r"C:\Users\Christian\AvianTech\config\birds.yaml",
    split   = "test",
    device  = 0,
    verbose = False,
)

yolo_map50    = metrics.box.map50
yolo_map5095  = metrics.box.map
yolo_precision = metrics.box.mp
yolo_recall    = metrics.box.mr
yolo_f1        = 2 * yolo_precision * yolo_recall / (yolo_precision + yolo_recall + 1e-8)

# ── Metrics WITH audio (Fusion scores from df_fused) ──────────────────────
# Fusion treats BirdNET confidence as a boost to detection confidence
# We measure: how many species have fusion_score >= 0.5 (reliable detection)
total_species   = len(df_fused)

fusion_avg      = df_fused["fusion_score"].mean()
fusion_precision = (df_fused["fusion_score"] >= 0.5).sum() / total_species
fusion_recall    = (df_fused["birdnet_conf"] > 0).sum() / total_species  # species BirdNET detected
fusion_f1        = 2 * fusion_precision * fusion_recall / (fusion_precision + fusion_recall + 1e-8)

# For mAP-equivalent: average of YOLO ap50 boosted by BirdNET where available
boosted_ap50   = (df_fused["yolo_ap50"] * 0.6 + df_fused["birdnet_conf"] * 0.4).mean()
boosted_ap5095 = boosted_ap50 * 0.65  # approximate scaling

# ── Data ──────────────────────────────────────────────────────────────────
metrics_labels = ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1 Score"]
without_audio  = [yolo_map50, yolo_map5095, yolo_precision, yolo_recall, yolo_f1]
with_audio     = [boosted_ap50, boosted_ap5095, fusion_precision, fusion_recall, fusion_f1]

# ══════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Grouped Bar Chart
# ══════════════════════════════════════════════════════════════════════════
x  = np.arange(len(metrics_labels))
w  = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2d3a")

bars1 = ax.bar(x - w/2, without_audio, width=w, color=ACC, label="Without Audio (YOLOv11 only)",
               edgecolor="#2a2d3a", zorder=3)
bars2 = ax.bar(x + w/2, with_audio,    width=w, color=GRN, label="With Audio (YOLOv11 + BirdNET)",
               edgecolor="#2a2d3a", zorder=3)

# Value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.1%}", ha="center", va="bottom",
            fontsize=9, color=ACC, fontweight="bold")
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.1%}", ha="center", va="bottom",
            fontsize=9, color=GRN, fontweight="bold")

# Improvement arrows
for i, (wo, wi) in enumerate(zip(without_audio, with_audio)):
    diff = wi - wo
    color = GRN if diff >= 0 else RED
    symbol = "▲" if diff >= 0 else "▼"
    ax.text(i, max(wo, wi) + 0.055, f"{symbol}{abs(diff)*100:.1f}%",
            ha="center", fontsize=9, color=color, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, color=WHT, fontsize=11)
ax.set_ylabel("Score", color=GRY, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title("AvianTech — Model Performance: Without Audio vs With Audio",
             color=WHT, fontsize=14, fontweight="bold", pad=14)
ax.tick_params(colors=GRY)
ax.grid(axis="y", color="#2a2d3a", linewidth=0.5, zorder=0)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=10)

plt.tight_layout()
out1 = os.path.join(save_dir, "comparison_bar_without_vs_with_audio.png")
plt.savefig(out1, dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"✅ Saved → {out1}")

# ══════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Radar Chart
# ══════════════════════════════════════════════════════════════════════════
from matplotlib.patches import FancyArrowPatch

angles = np.linspace(0, 2 * np.pi, len(metrics_labels), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

vals_wo = without_audio + without_audio[:1]
vals_wi = with_audio    + with_audio[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor(BG)
ax.set_facecolor(CARD)

ax.plot(angles, vals_wo, color=ACC, linewidth=2.5, label="Without Audio")
ax.fill(angles, vals_wo, color=ACC, alpha=0.15)
ax.plot(angles, vals_wi, color=GRN, linewidth=2.5, label="With Audio")
ax.fill(angles, vals_wi, color=GRN, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_labels, color=WHT, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["20%", "40%", "60%", "80%", "100%"], color=GRY, fontsize=8)
ax.grid(color="#2a2d3a", linewidth=0.8)
ax.spines["polar"].set_edgecolor("#2a2d3a")
ax.set_title("AvianTech — Radar: Without Audio vs With Audio",
             color=WHT, fontsize=13, fontweight="bold", pad=20)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT,
          fontsize=10, loc="upper right", bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
out2 = os.path.join(save_dir, "comparison_radar_without_vs_with_audio.png")
plt.savefig(out2, dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"✅ Saved → {out2}")

# ══════════════════════════════════════════════════════════════════════════
# FIGURE 3 — KPI Summary Cards: Without vs With Audio
# ══════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(16, 5))
fig.patch.set_facecolor(BG)

fig.text(0.5, 1.0, "AvianTech — KPI Summary: Without Audio vs With Audio",
         ha="center", fontsize=15, fontweight="bold", color=WHT)

for i, (label, wo, wi) in enumerate(zip(metrics_labels, without_audio, with_audio)):
    diff   = wi - wo
    d_col  = GRN if diff >= 0 else RED
    d_sym  = "▲" if diff >= 0 else "▼"

    # Without audio card (left)
    ax1 = fig.add_axes([0.01 + i * 0.196, 0.52, 0.185, 0.38])
    ax1.set_facecolor(CARD)
    for sp in ax1.spines.values():
        sp.set_edgecolor(ACC); sp.set_linewidth(2)
    ax1.set_xticks([]); ax1.set_yticks([])
    ax1.text(0.5, 0.72, f"{wo:.1%}", transform=ax1.transAxes,
             ha="center", fontsize=18, fontweight="bold", color=ACC)
    ax1.text(0.5, 0.30, label,       transform=ax1.transAxes,
             ha="center", fontsize=9,  fontweight="bold", color=WHT)
    ax1.text(0.5, 0.05, "No Audio",  transform=ax1.transAxes,
             ha="center", fontsize=7.5, color=GRY)

    # With audio card (right)
    ax2 = fig.add_axes([0.01 + i * 0.196, 0.08, 0.185, 0.38])
    ax2.set_facecolor(CARD)
    for sp in ax2.spines.values():
        sp.set_edgecolor(GRN); sp.set_linewidth(2)
    ax2.set_xticks([]); ax2.set_yticks([])
    ax2.text(0.5, 0.72, f"{wi:.1%}",            transform=ax2.transAxes,
             ha="center", fontsize=18, fontweight="bold", color=GRN)
    ax2.text(0.5, 0.30, f"{d_sym} {abs(diff)*100:.1f}%", transform=ax2.transAxes,
             ha="center", fontsize=10, fontweight="bold", color=d_col)
    ax2.text(0.5, 0.05, "With Audio", transform=ax2.transAxes,
             ha="center", fontsize=7.5, color=GRY)

plt.tight_layout()
out3 = os.path.join(save_dir, "comparison_kpi_without_vs_with_audio.png")
plt.savefig(out3, dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"✅ Saved → {out3}")

# ══════════════════════════════════════════════════════════════════════════
# PRINT SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "═"*58)
print(f"{'Metric':<18} {'Without Audio':>14} {'With Audio':>14} {'Δ':>8}")
print("═"*58)
for label, wo, wi in zip(metrics_labels, without_audio, with_audio):
    diff = wi - wo
    sym  = "▲" if diff >= 0 else "▼"
    print(f"  {label:<16} {wo:>13.2%} {wi:>13.2%} {sym}{abs(diff)*100:>5.2f}%")
print("═"*58)
print(f"\n✅ 3 comparison charts saved to:\n   {save_dir}")

Ultralytics 8.4.21  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11s summary (fused): 101 layers, 9,427,119 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 334.0416.2 MB/s, size: 111.4 KB)
val: Scanning C:\Users\Christian\AvianTech\data\datasets\test\labels.cache... 185 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 185/185  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 3.5it/s 3.5s0.3s
                   all        185        290      0.906      0.775      0.855      0.746
Speed: 2.1ms preprocess, 9.7ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to C:\Users\Christian\AvianTech\runs\detect\val7


<Figure size 1300x600 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v11\comparison_bar_without_vs_with_audio.png


<Figure size 800x800 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v11\comparison_radar_without_vs_with_audio.png


C:\Users\Christian\AppData\Local\Temp\ipykernel_12984\2051374807.py:185: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


<Figure size 1600x500 with 10 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v11\comparison_kpi_without_vs_with_audio.png

══════════════════════════════════════════════════════════
Metric              Without Audio     With Audio        Δ
══════════════════════════════════════════════════════════
  mAP@0.5                 85.52%        51.31% ▼34.21%
  mAP@0.5:0.95            74.55%        33.35% ▼41.20%
  Precision               90.57%        56.76% ▼33.82%
  Recall                  77.55%         0.00% ▼77.55%
  F1 Score                83.56%         0.00% ▼83.56%
══════════════════════════════════════════════════════════

✅ 3 comparison charts saved to:
   C:\Users\Christian\AvianTech\results_charts_v11
